# AI Handwriting Generation — AC-GAN Training
**Step 0:** Runtime > Change runtime type > **T4 GPU** before running anything.

In [ ]:
# Cell 1: Confirm GPU
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: No GPU found — go to Runtime > Change runtime type > T4 GPU')

In [ ]:
# Cell 2: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 3: Clone or update repo + install deps
import os

REPO = 'AI-Powered-Handwriting-Generation-System'
REPO_PATH = f'/content/{REPO}'

if os.path.exists(REPO_PATH):
    print('Repo exists — pulling latest changes...')
    os.system(f'git -C {REPO_PATH} pull')
else:
    print('Cloning repo...')
    os.system(f'git clone --depth 1 https://github.com/Abdullahshaz70/{REPO}.git {REPO_PATH}')

os.chdir(f'{REPO_PATH}/Data')
print('Working dir:', os.getcwd())

!pip install fpdf2 -q
print('Dependencies installed.')

In [ ]:
# Cell 4: Copy Writers_pngs from Drive
# IMPORTANT: Your Writers_pngs folder must be in Google Drive root (My Drive/Writers_pngs/)
import shutil, os

src  = '/content/drive/MyDrive/Writers_pngs'
dest = '/content/AI-Powered-Handwriting-Generation-System/Data/Writers_pngs'

if os.path.exists(dest):
    print('Writers_pngs already present — skipping copy')
elif os.path.exists(src):
    shutil.copytree(src, dest)
    print('Writers_pngs copied successfully')
else:
    print('ERROR: Writers_pngs not found at', src)
    print('Upload your Writers_pngs folder to Google Drive root first.')
    raise FileNotFoundError(src)

# Show what writers were loaded
skip = {'Writers_Zip', 'output_preview'}
total = 0
for d in sorted(os.listdir(dest)):
    if d in skip: continue
    p = os.path.join(dest, d)
    if os.path.isdir(p):
        n = len([f for f in os.listdir(p) if f.endswith('.png')])
        print(f'  {d}: {n} samples')
        total += n
print(f'Total: {total} samples')

In [ ]:
# Cell 5: Clear old checkpoints and train from scratch
import shutil, os

ckpt_dir = '/content/AI-Powered-Handwriting-Generation-System/checkpoints'
if os.path.exists(ckpt_dir):
    shutil.rmtree(ckpt_dir)
    print('Old checkpoints cleared')
os.makedirs(ckpt_dir, exist_ok=True)

os.chdir('/content/AI-Powered-Handwriting-Generation-System/Data')
!python train.py

In [ ]:
# Cell 6: Save best_model.pt to Drive AND download it
import shutil, os
from google.colab import files

ckpt_src  = '/content/AI-Powered-Handwriting-Generation-System/checkpoints/best_model.pt'
drive_dir = '/content/drive/MyDrive/handwriting_checkpoints'
os.makedirs(drive_dir, exist_ok=True)

shutil.copy(ckpt_src, os.path.join(drive_dir, 'best_model.pt'))
print('Saved to Drive:', drive_dir)

# Download directly to your computer
files.download(ckpt_src)
print('Download started — save best_model.pt into your project checkpoints/ folder')

In [ ]:
# Cell 7: Generate handwritten PDF with trained model
os.chdir('/content/AI-Powered-Handwriting-Generation-System/Data')
!python renderer.py

In [ ]:
# Cell 8: Save PDF to Drive + download
import shutil
from google.colab import files

pdf_src = '/content/AI-Powered-Handwriting-Generation-System/output_handwritten.pdf'
shutil.copy(pdf_src, '/content/drive/MyDrive/output_handwritten.pdf')
print('PDF saved to Drive')

files.download(pdf_src)
print('PDF download started')